<a href="https://colab.research.google.com/github/ABROCK1516/Ukraine-Losses-Weather/blob/main/Ukraine_Losses_Weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Russian Equipment & Personnel Losses & Front-line Weather**
## *Data Acquisition and Preprocessing – Individual project*
**Alex Brock | C23426432 | TU912/4**

## **Introduction**
This project combines daily figures for Russian equipment and personnel losses in the war in Ukraine with weather observations from near the front line. Weather is often said to shape the fighting, for example mud seasons slowing armoured vehicles, or wind and fog grounding FPV drones which allows for more ground assaults and offensive actions. This project tests whether those patterns can be seen in the data.

I chose this topic because I have been following the war closely since the full scale invasion of Ukraine in 2022. I keep up to date with the war through YouTube channels and the news. From the start of 2023 onwards, I have watched and read about both sides of the war use weather patterns and conditions to their advantage such as fog to aid in assaulting dug in positions.

I was particularly interested in the military side: how weapons and drones are used, and how much outside conditions like the weather affect them. Commentary on the war often claims that weather changes the fighting, and I wanted to see whether the data actually supports that.

## **Research questions**
The project is built in three layers, each adding a dataset and new questions:

1. **Kaggle data alone:** How have daily losses changed over the course of the war? Has the mix of equipment lost changed, and has the ratio of personnel to vehicles lost changed over time?
2. **Adding weather:** Do daily losses vary with weather conditions such as rain, freezing temperatures, wind and low visibility? Which loss categories are most affected?
3. **Adding combat engagements (extension):** When losses change with the weather, is that because there is less fighting, or because the fighting causes fewer losses?

## **Data sources**

| Source | What it contains | Period | How it was obtained | Role |
|---|---|---|---|---|
| [Kaggle: 2022 Russia Ukraine War (piterfm)](https://www.kaggle.com/datasets/piterfm/2022-ukraine-russian-war) | Daily cumulative Russian equipment losses by category, and personnel losses | 25 Feb 2022 – 10 Oct 2026 (1,689 rows) | Downloaded as two CSV files | Main dataset |
| [pogodaiklimat.ru](http://www.pogodaiklimat.ru/) | 3-hourly weather observations: wind, visibility, cloud cover, weather phenomena, temperature, precipitation | Jan 2023 – Oct 2026 | Web scraping (urllib and BeautifulSoup) | Complementary dataset |
| [ukraine.vefgreining.com](https://ukraine.vefgreining.com/) | Daily number of combat engagements reported by Ukraine's General Staff | 22 Feb 2023 – 16 Oct 2026 | JSON data file | Extension |

Weather is taken from two stations: **occupied Donetsk** (station 34519), which is close to most of the heaviest fighting since 2023, and **Kharkiv** (station 34300), in the Kharkiv region, where fighting continues around Kupiansk, it is used as a second station to check whether results hold.

## **Notes on the data**
- **These are claimed figures.** The loss and engagement figures are reported by Ukraine and have not been independently verified.
- **Personnel figures are estimates.** The dataset itself marks them as approximate (“about” or “more”).
- **"Occupied Donetsk".** The Donetsk weather station is in Russian-occupied territory, and the weather site lists it under Russia.
- **Weather coverage.** The weather site has little data for 2022, so the weather analysis covers 2023 onwards. The Kaggle data is kept in full for Step 1.
- **One weather point for a long front.** The front line is over 1,000 km long, so weather at one station is only an approximation of conditions where the fighting took place.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Import the os library, which lets Python work with folders and file paths
import os
os.chdir("/content/drive/MyDrive/Data ACQ - Project/Data")
!pwd

/content/drive/MyDrive/Data ACQ - Project/Data


## **Step 1: Import the Kaggle dataset**
The dataset is "2022 Russia Ukraine War" by piterfm on Kaggle. It has two CSV files:
running totals of Russian equipment losses and daily personnel losses.

**About the dataset**
The main dataset is ["2022 Russia Ukraine War" by piterfm](https://www.kaggle.com/datasets/piterfm/2022-ukraine-russian-war) on Kaggle. It records Russian losses in the war as reported by official Ukrainian sources, and has two CSV files:
- `russia_losses_equipment.csv` (1,689 rows × 20 columns): losses by equipment category, such as tanks, armoured personnel carriers (APCs), artillery and drones.
- `russia_losses_personnel.csv` (1,689 rows × 5 columns): personnel losses and prisoners of war (POW).

Both files cover 25 February 2022 to 10 October 2026, with one row per day, and have the same dates, so they can be joined on the `date` column. The figures are **cumulative**: each row gives the total lost since the start of the invasion, not the losses on that day, so daily losses have to be calculated. The data was downloaded on 16 October 2026. Because the dataset is updated weekly, it is not downloaded again, so the results stay reproducible.

In [4]:
# Import pandas, the library used to load and work with tables of data
import pandas as pd

# Load the equipment losses CSV into a DataFrame (a table) called equipment
equipment = pd.read_csv("russia_losses_equipment.csv")

# Load the personnel losses CSV into a DataFrame called personnel
personnel = pd.read_csv("russia_losses_personnel.csv")

In [5]:
# Check the size of the equipment table: (number of rows, number of columns)
equipment.shape

(1689, 20)

In [6]:
# Show the first 10 rows, to see what the data looks like
equipment.head(10)

,date,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,drone,naval ship,anti-aircraft warfare,special equipment,mobile SRBM system,greatest losses direction,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems
0,2026-10-10,1690,443,356,12406,25515,51000,2201,NaN,NaN,556926,35,1699,4812.0,NaN,NaN,160946.0,5126.0,2.0,2917.0
1,2026-10-09,1689,443,356,12402,25509,50950,2199,NaN,NaN,555335,35,1693,4810.0,NaN,NaN,160412.0,5126.0,2.0,2908.0
2,2026-10-08,1688,443,356,12399,25502,50883,2194,NaN,NaN,553578,35,1692,4806.0,NaN,NaN,159795.0,5126.0,2.0,2890.0
3,2026-10-07,1687,443,356,12396,25498,50836,2192,NaN,NaN,551876,35,1690,4800.0,NaN,NaN,159128.0,5126.0,2.0,2878.0
4,2026-10-06,1686,443,356,12396,25494,50785,2184,NaN,NaN,549986,35,1688,4795.0,NaN,NaN,158551.0,5126.0,2.0,2866.0
5,2026-10-05,1685,443,356,12393,25487,50733,2182,NaN,NaN,548095,35,1688,4793.0,NaN,NaN,157985.0,5126.0,2.0,2859.0
6,2026-10-04,1684,443,356,12391,25483,50686,2179,NaN,NaN,546240,35,1686,4788.0,NaN,NaN,157258.0,5126.0,2.0,2837.0
7,2026-10-03,1683,443,356,12391,25477,50637,2178,NaN,NaN,544498,35,1683,4783.0,NaN,NaN,156731.0,5126.0,2.0,2824.0
8,2026-10-02,1682,443,356,12388,25471,50592,2175,NaN,NaN,542853,35,1683,4780.0,NaN,NaN,156309.0,5126.0,2.0,2810.0
9,2026-10-01,1681,443,356,12388,25464,50555,2174,NaN,NaN,541383,35,1683,4774.0,NaN,NaN,155887.0,5126.0,2.0,2803.0


In [7]:
# Show the column names and the data type of each column
equipment.dtypes

,0
date,object
day,int64
aircraft,int64
helicopter,int64
tank,int64
APC,int64
field artillery,int64
MRL,int64
military auto,float64
fuel tank,float64


In [8]:
# Summary statistics for each numeric column:
equipment.describe()

,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,drone,naval ship,anti-aircraft warfare,special equipment,mobile SRBM system,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems
count,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,65.000000,65.000000,1689.000000,1689.000000,1689.000000,1670.000000,36.000000,1624.000000,1624.000000,1123.000000,161.000000
mean,846.000000,343.914742,303.917703,7302.714624,14825.561279,17909.660746,1046.749556,1047.507692,69.323077,66177.216104,23.708111,788.043813,2282.670060,3.944444,37143.435961,2428.223522,1.266251,2036.956522
std,487.716619,79.985283,60.524649,3836.696267,7956.872953,15393.569483,548.811027,466.162060,7.545917,125776.097509,7.141723,473.888851,1695.212102,0.333333,38493.183090,1501.023468,0.442194,475.056514
min,2.000000,10.000000,7.000000,80.000000,516.000000,49.000000,4.000000,100.000000,60.000000,0.000000,2.000000,0.000000,10.000000,2.000000,1796.000000,84.000000,1.000000,1306.000000
25%,424.000000,308.000000,294.000000,3675.000000,7131.000000,2837.000000,539.000000,600.000000,60.000000,2402.000000,18.000000,289.000000,381.000000,4.000000,6452.750000,1181.250000,1.000000,1636.000000
50%,846.000000,359.000000,326.000000,7974.000000,15307.000000,13959.000000,1104.000000,1178.000000,73.000000,11187.000000,28.000000,853.000000,2425.500000,4.000000,21002.000000,2401.000000,1.000000,1981.000000
75%,1268.000000,421.000000,340.000000,11104.000000,23130.000000,31458.000000,1466.000000,1437.000000,76.000000,51043.000000,28.000000,1207.000000,3942.750000,4.000000,60242.750000,3635.500000,2.000000,2442.000000
max,1690.000000,443.000000,356.000000,12406.000000,25515.000000,51000.000000,2201.000000,1701.000000,76.000000,556926.000000,35.000000,1699.000000,4812.000000,4.000000,160946.000000,5126.000000,2.000000,2917.000000


In [9]:
# Count the missing (empty) values in each column
equipment.isnull().sum()

,0
date,0
day,0
aircraft,0
helicopter,0
tank,0
APC,0
field artillery,0
MRL,0
military auto,1624
fuel tank,1624


In [10]:
# Check the size of the personnel table: (number of rows, number of columns)
personnel.shape

(1689, 5)

In [11]:
# Show the first 5 rows, to see what the data looks like
personnel.head()

,date,day,personnel,personnel*,POW
0,2026-10-10,1690,1550910,about,NaN
1,2026-10-09,1689,1549490,about,NaN
2,2026-10-08,1688,1547780,about,NaN
3,2026-10-07,1687,1545950,about,NaN
4,2026-10-06,1686,1544260,about,NaN


In [12]:
# Show the column names and the data type of each column
personnel.dtypes

,0
date,object
day,int64
personnel,int64
personnel*,object
POW,float64


In [13]:
# Summary statistics for each numeric column:
personnel.describe()

,day,personnel,POW
count,1689.000000,1.689000e+03,62.000000
mean,846.000000,6.250922e+05,386.387097
std,487.716619,4.793099e+05,131.440363
min,2.000000,2.800000e+03,0.000000
25%,424.000000,1.864200e+05,389.000000
50%,846.000000,5.286200e+05,421.000000
75%,1268.000000,1.067100e+06,474.500000
max,1690.000000,1.550910e+06,496.000000


In [14]:
# Count the missing (empty) values in each column
personnel.isnull().sum()

,0
date,0
day,0
personnel,0
personnel*,0
POW,1627


In [15]:
# Compare the date columns row by row; .all() is True only if every row matches
(equipment["date"] == personnel["date"]).all()

np.True_

In [16]:
# Columns with lots of missing values, to check when each one has data
columns_to_check = ["military auto", "fuel tank", "vehicles and fuel tanks",
                    "cruise missiles", "submarines", "ground robotic systems",
                    "mobile SRBM system", "greatest losses direction"]

# Loop through each column name in the list
for col in columns_to_check:
    # Keep only the rows where this column is NOT empty
    filled = equipment[equipment[col].notnull()]
    # Print the column name, first date with data, last date with data, and how many rows have data
    print(col, "|", filled["date"].min(), "to", filled["date"].max(), "|", len(filled), "rows")

military auto | 2022-02-25 to 2022-04-30 | 65 rows
fuel tank | 2022-02-25 to 2022-04-30 | 65 rows
vehicles and fuel tanks | 2022-05-01 to 2026-10-10 | 1624 rows
cruise missiles | 2022-05-01 to 2026-10-10 | 1624 rows
submarines | 2023-09-14 to 2026-10-10 | 1123 rows
ground robotic systems | 2026-05-03 to 2026-10-10 | 161 rows
mobile SRBM system | 2022-03-26 to 2022-04-30 | 36 rows
greatest losses direction | 2022-04-25 to 2023-01-02 | 203 rows


In [17]:
# Keep only the rows where POW has a value
pow_filled = personnel[personnel["POW"].notnull()]
# Print the first date, last date and number of rows with a POW value
print(pow_filled["date"].min(), "to", pow_filled["date"].max(), "|", len(pow_filled), "rows")

2022-02-25 to 2022-04-27 | 62 rows


## **Observaions**
**Structure:** Each row is one day. The numbers are running totals since the start of the invasion, not the losses on that day. The equipment file has 1,689 rows and 20 columns, and the personnel file has 1,689 rows and 5 columns. I checked that both files have the same date in every row (the check returned `True`), so they can be joined on `date`.

**Data types:** The `date` column is stored as `object`, which means pandas sees the dates as text. It needs converting to a real date before it can be used for date calculations. Some columns of whole numbers are stored as floats (decimals). This is because these columns have empty values, and pandas cannot store an empty value in a whole-number column.

**Values:**
- The `day` column starts at 2. The invasion began on 24 Feb 2022 (day 1), so the first row (25 Feb) already includes two days of losses.
- Because the numbers are running totals, the minimum of each column is just its first value. For example, 516 APCs had already been claimed by the first row.
- Drones start at 0. This makes sense, as drones were not widely used early in the war.
- Some new categories start with a high number, such as `cruise missiles` (84) and `ground robotic systems` (1,306). This means earlier losses were added in one go on the first day. Since ground robotic systems were first reported (3 May 2026), about 10 per day have been claimed.

**Missing values:** The empty values are not errors in the data. They happen because the categories in the daily report changed over time:
- **Replaced:** `military auto` and `fuel tank` stop on 30 Apr 2022, and `vehicles and fuel tanks` starts the next day. Together they fill every row (65 + 1,624 = 1,689), so the two old categories were combined into one.
- **Added later:** `cruise missiles` (1 May 2022), `submarines` (14 Sep 2023) and `ground robotic systems` (3 May 2026). Submarines first appear the day after Ukraine's 13 Sep 2023 missile strike on the submarine Rostov-on-Don in Sevastopol, which fits the reports being published the next morning.
- **Dropped:** `mobile SRBM system` was only reported from 26 Mar to 30 Apr 2022. `greatest losses direction` contains place names, not numbers, and was only reported from 25 Apr 2022 to 2 Jan 2023, with gaps, so it is not used.
- **Personnel:** `POW` was only reported from 25 Feb to 27 Apr 2022. After that it stopped being published, possibly because Ukraine chose not to share prisoner numbers.

This means a category can only be compared over time for the period it was reported.

